# DBP Homework #9

In [ ]:
!pip install pyyaml pandas matplotlib tqdm "elasticsearch==8.10.1"

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.elasticsearch_client import ElasticsearchClient
from data.index_mapping import mapping, movie_mapping

INDEX_NAME = "movie_reviews_homework9"
MOVIE_INDEX = "movies_homework9"
TOP_K = 100
QUERY_ALL = {"query": {"match_all": {}}}

In [ ]:
es_client = ElasticsearchClient()
print(es_client.client.info())

In [ ]:
with open("./data/movie_data_big.json", "r") as f:
    movie_data = json.load(f)
print(len(movie_data))  # 100789

### 제공되는 공통 함수

In [ ]:
def reset_index():
    """리뷰 인덱스를 지우고 다시 생성"""
    es_client.delete_index(index_name=INDEX_NAME)
    es_client.create_index(index_name=INDEX_NAME, mapping=mapping)

def make_actions(docs):
    """bulk 요청용 action 리스트 생성 (_id = 리스트 순서)"""
    return [
        {"_op_type": "index", "_index": INDEX_NAME, "_id": i, "_source": doc}
        for i, doc in enumerate(docs)
    ]

def measure(fn, repeat=1, setup=None):
    """setup() 후 fn()을 repeat번 실행하고 (평균 소요 시간, 마지막 결과)를 반환"""
    times, result = [], None
    for _ in range(repeat):
        if setup is not None:
            setup()
        start = time.time()
        result = fn()
        times.append(time.time() - start)
    return float(np.mean(times)), result

## **[5점]** 리뷰 10,000건을 insert_one_document로 하나씩 넣는 경우와 bulk_request로 넣는 경우의 소요 시간과 처리량을 측정하기

In [ ]:
N_SUBSET = 10000
subset = movie_data[:N_SUBSET]

###### EDIT HERE ######
def insert_one_by_one():
    # subset의 문서를 insert_one_document로 하나씩 삽입 (doc_id = 리스트 순서)
    pass

def insert_with_bulk():
    # make_actions()로 action 리스트를 만들어 bulk_request로 삽입하고, 그 결과를 반환
    pass
###### EDIT HERE ######

t_single, _ = measure(insert_one_by_one, repeat=1, setup=reset_index)
print(f"[하나씩] {t_single:.2f} s, count = {es_client.count(index_name=INDEX_NAME)}")  # count should be 10000

t_bulk, (success, errors) = measure(insert_with_bulk, repeat=3, setup=reset_index)
print(f"[bulk]   {t_bulk:.2f} s, count = {es_client.count(index_name=INDEX_NAME)}, errors = {len(errors)}")  # 10000, 0

In [ ]:
result_1a1 = pd.DataFrame({
    "방식": ["insert_one_document", "bulk"],
    "소요 시간 (s)": [t_single, t_bulk],
    "처리량 (docs/s)": [N_SUBSET / t_single, N_SUBSET / t_bulk],
})
result_1a1.round(2)

## **[5점]** 전체 리뷰(100,789건)를 chunk_size 100, 500, 2000, 5000으로 bulk 적재하여 소요 시간을 측정하기

In [ ]:
CHUNK_SIZES = [100, 500, 2000, 5000]
chunk_times = []

###### EDIT HERE ######
# CHUNK_SIZES의 각 값에 대해 전체 리뷰(movie_data)를 bulk 적재하는 시간을 measure()로 측정 (repeat=3, setup=reset_index)
# 측정한 시간은 chunk_times에 추가하고, chunk_size / 소요 시간 / count / errors 개수를 출력

###### EDIT HERE ######

In [ ]:
result_1a2 = pd.DataFrame({
    "chunk_size": CHUNK_SIZES,
    "요청 횟수": [-(-len(movie_data) // c) for c in CHUNK_SIZES],
    "소요 시간 (s)": chunk_times,
    "처리량 (docs/s)": [len(movie_data) / t for t in chunk_times],
})
result_1a2.round(2)

## **[5점]** match_all로 전체 문서를 조회할 때, page size 10, 100, 1000 각각에 대해 search + scroll 반복 방식과 scan 방식의 소요 시간을 측정하기

In [ ]:
PAGE_SIZES = [10, 100, 1000]
print(es_client.count(index_name=INDEX_NAME))  # 100789

In [ ]:
###### EDIT HERE ######
def scroll_all(page_size):
    """search(scroll=...) 후 scroll을 반복하여 전체 문서 수를 반환"""
    count = 0
    return count

def scan_all(page_size):
    """scan_index로 전체 문서 수를 반환"""
    count = 0
    return count
###### EDIT HERE ######

scroll_times, scan_times = [], []
for page_size in PAGE_SIZES:
    t_scroll, n_scroll = measure(lambda: scroll_all(page_size), repeat=3)
    t_scan, n_scan = measure(lambda: scan_all(page_size), repeat=3)
    scroll_times.append(t_scroll)
    scan_times.append(t_scan)
    print(f"page_size = {page_size:4d} | scroll: {t_scroll:.2f} s ({n_scroll}) | scan: {t_scan:.2f} s ({n_scan})")  # 100789

In [ ]:
result_1b1 = pd.DataFrame({
    "page_size": PAGE_SIZES,
    "요청 횟수 (약)": [-(-100789 // p) for p in PAGE_SIZES],
    "search + scroll (s)": scroll_times,
    "scan (s)": scan_times,
})
result_1b1.round(2)

## **[5점]** 일반 search에서 from=10000, size=10으로 요청한 결과를 확인하기

In [ ]:
try:
    resp = es_client.client.search(index=INDEX_NAME, query={"match_all": {}}, from_=10000, size=10)
    print(len(resp["hits"]["hits"]))
except Exception as e:
    print(type(e).__name__, ":", e)

### 제공되는 데이터 적재 함수

In [ ]:
with open("./data/movies_emb.json", "r") as f:
    movies = json.load(f)
with open("./data/queries_emb.json", "r") as f:
    queries = json.load(f)
print(len(movies), len(queries))  # 9452 33

TITLE = {m["movieId"]: m["title"] for m in movies}
TYPES = ["exact", "keyword", "semantic", "mixed"]
Q = {q["query_id"]: q for q in queries}

In [ ]:
es_client.delete_index(index_name=MOVIE_INDEX)
es_client.create_index(index_name=MOVIE_INDEX, mapping=movie_mapping)

movie_actions = [
    {"_op_type": "index", "_index": MOVIE_INDEX, "_id": m["movieId"], "_source": m}
    for m in movies
]
success, errors = es_client.bulk_request(actions=movie_actions, chunk_size=500)
print(success, len(errors), es_client.count(index_name=MOVIE_INDEX))  # 9452 0 9452

### 제공되는 평가 함수

In [ ]:
def hits_to_ids(hits):
    """검색 결과(hits)를 movieId 리스트로 변환 (_id = movieId)"""
    return [int(h["_id"]) for h in hits]

def rank_of(q, ids):
    """정답 영화 중 가장 높은 순위 (1부터). 없으면 None"""
    relevant = set(q["relevant"])
    for i, mid in enumerate(ids):
        if mid in relevant:
            return i + 1
    return None

def evaluate(search_fn):
    """모든 질문에 search_fn을 적용하여 질문별 순위, RR, Hit@1, Hit@10을 반환"""
    rows = []
    for q in queries:
        r = rank_of(q, search_fn(q))
        rows.append({
            "query_id": q["query_id"],
            "type": q["type"],
            "rank": r,
            "RR": 1.0 / r if r else 0.0,
            "Hit@1": float(r is not None and r <= 1),
            "Hit@10": float(r is not None and r <= 10),
        })
    return pd.DataFrame(rows)

def by_type(df, metric="RR"):
    """유형별 + 전체(ALL) 평균"""
    s = df.groupby("type")[metric].mean().reindex(TYPES)
    s["ALL"] = df[metric].mean()
    return s

def summarize(results):
    """{방식 이름: evaluate 결과} -> 유형별 MRR, Hit@1, Hit@10 표"""
    table = {}
    for metric, label in [("RR", "MRR"), ("Hit@1", "Hit@1"), ("Hit@10", "Hit@10")]:
        for name, df in results.items():
            table[(label, name)] = by_type(df, metric)
    out = pd.DataFrame(table)
    out.index.name = "type"
    return out.round(3)

def show_top(q, search_fn, n=5):
    """상위 n개 결과 출력 (정답은 * 표시)"""
    relevant = set(q["relevant"])
    print(f"[{q['query_id']}] {q['query']}")
    for i, mid in enumerate(search_fn(q)[:n]):
        mark = "*" if mid in relevant else " "
        print(f"  {mark} {i + 1}. {TITLE.get(mid, mid)}")

def rrf(ranked_lists, k=60, size=TOP_K):
    """여러 순위 리스트를 Reciprocal Rank Fusion으로 합침: score = Σ 1 / (k + rank)"""
    scores = {}
    for ids in ranked_lists:
        for rank, mid in enumerate(ids, start=1):
            scores[mid] = scores.get(mid, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)[:size]

## **[8점]** title과 overview를 대상으로 하는 BM25 검색(multi_match)과, embedding을 대상으로 하는 kNN 검색 함수를 작성하기

In [ ]:
###### EDIT HERE ######
def bm25_search(q, size=TOP_K):
    # title과 overview를 대상으로 multi_match 검색, 상위 size개의 movieId 리스트를 반환
    query = {
    }
    return hits_to_ids(es_client.search(query=query, index_name=MOVIE_INDEX))

def knn_search(q, size=TOP_K):
    # embedding 필드를 대상으로 kNN 검색, 상위 size개의 movieId 리스트를 반환
    query = {
    }
    return hits_to_ids(es_client.search(query=query, index_name=MOVIE_INDEX))
###### EDIT HERE ######

res_bm25 = evaluate(bm25_search)
res_knn = evaluate(knn_search)
summarize({"BM25": res_bm25, "kNN": res_knn})

## **[10점]** 세 질문에 대해 BM25와 kNN의 정답 순위를 확인하고, show_top()으로 확인한 실제 상위 결과를 근거로 각 결과가 나온 원인을 설명하기

In [ ]:
def fmt_rank(r):
    return f"{r}위" if r else f"{TOP_K}위 밖"

for qid in ["e04", "s06", "s05"]:
    q = Q[qid]
    print(f"===== {qid}: BM25 {fmt_rank(rank_of(q, bm25_search(q)))}, kNN {fmt_rank(rank_of(q, knn_search(q)))}")
    print("--- BM25")
    show_top(q, bm25_search)
    print("--- kNN")
    show_top(q, knn_search)
    print()

## **[12점]** BM25와 kNN을 결합하는 두 가지 하이브리드 검색을 작성하고, 각 방식에서 성능(전체 MRR)이 가장 좋은 파라미터를 직접 찾기

### 제공되는 점수 범위 확인 함수

In [ ]:
def top_scores(query, n=5):
    return [round(h["_score"], 3) for h in es_client.search(query=query, index_name=MOVIE_INDEX)[:n]]

for qid in ["e01", "e03", "s06"]:
    q = Q[qid]
    bm = {"query": {"multi_match": {"query": q["query"], "fields": ["title", "overview"]}}, "size": 5, "_source": False}
    kn = {"knn": {"field": "embedding", "query_vector": q["embedding"], "k": 5, "num_candidates": 1000}, "size": 5, "_source": False}
    print(f"[{qid}] BM25: {top_scores(bm)} | kNN: {top_scores(kn)}")

#### (1) 점수 합산

In [ ]:
###### EDIT HERE ######
def hybrid_sum_search(q, knn_boost, size=TOP_K):
    # 한 번의 검색 요청에 BM25 검색(query, boost=1)과 kNN 검색(knn, boost=knn_boost)을 함께 넣어 검색
    query = {
    }
    return hits_to_ids(es_client.search(query=query, index_name=MOVIE_INDEX))

BOOSTS = []  # 탐색할 kNN boost 값 (범위와 간격은 직접 정할 것)
###### EDIT HERE ######

sum_mrr = pd.DataFrame({
    float(b): by_type(evaluate(lambda q: hybrid_sum_search(q, b))) for b in BOOSTS
}).T

print("[점수 합산] kNN boost별 MRR")
sum_mrr.round(3)

#### (2) RRF

In [ ]:
# BM25, kNN 결과는 파라미터와 무관하므로 한 번만 검색해 둠
bm25_cache = {q["query_id"]: bm25_search(q) for q in queries}
knn_cache = {q["query_id"]: knn_search(q) for q in queries}

###### EDIT HERE ######
def hybrid_rrf_search(q, k):
    # bm25_cache와 knn_cache의 결과를 rrf()로 합쳐 반환
    pass

RRF_KS = []  # 탐색할 RRF k 값 (범위와 간격은 직접 정할 것)
###### EDIT HERE ######

rrf_mrr = pd.DataFrame({
    k: by_type(evaluate(lambda q: hybrid_rrf_search(q, k))) for k in RRF_KS
}).T

print("[RRF] k별 MRR")
rrf_mrr.round(3)

#### 그래프

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, table, title, xlabel in [
    (axes[0], sum_mrr, "Score sum", "kNN boost (BM25 boost = 1)"),
    (axes[1], rrf_mrr, "RRF", "RRF k"),
]:
    for col in TYPES + ["ALL"]:
        style = {"linewidth": 3} if col == "ALL" else {}
        ax.plot(table.index, table[col], marker="o", label=col, **style)
    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("MRR")
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

#### 최적 파라미터와 최종 비교

In [ ]:
best_boost = sum_mrr["ALL"].idxmax()
best_k = rrf_mrr["ALL"].idxmax()
print(f"best kNN boost = {best_boost}, best RRF k = {best_k}")

res_sum = evaluate(lambda q: hybrid_sum_search(q, best_boost))
res_rrf = evaluate(lambda q: hybrid_rrf_search(q, best_k))
summarize({
    "BM25": res_bm25,
    "kNN": res_knn,
    f"Sum(1:{best_boost})": res_sum,
    f"RRF(k={best_k})": res_rrf,
})